In [1]:
# RNN multiclass

In [5]:
import re
import torch
import emoji
import torch.nn as nn
import pandas as pd
import json
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import classification_report

# Load datasets
train_df = pd.read_csv("en_train.csv")
dev_df = pd.read_csv("en_dev.csv")

# Emoji and text cleaning
def convert_emojis(text):
    return emoji.demojize(text, delimiters=(" ", " "))

def clean_text(text):
    text = re.sub(r"#USER#", "", text)
    text = convert_emojis(text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', 'USER_MENTION', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

train_df["clean_text"] = train_df["text"].apply(clean_text)
dev_df["clean_text"] = dev_df["text"].apply(clean_text)

# Multiclass labels
label_map = {
    "Generalized Hope": 0,
    "Realistic Hope": 1,
    "Unrealistic Hope": 2,
    "Not Hope": 3,
    "Sarcasm": 4
}
train_df["label"] = train_df["multiclass"].map(label_map)
dev_df["label"] = dev_df["multiclass"].map(label_map)

# Vocabulary
word_counts = Counter()
for sentence in train_df["clean_text"]:
    word_counts.update(sentence.split())

vocab = {word: idx + 2 for idx, (word, _) in enumerate(word_counts.most_common())}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

def text_to_tensor(text, vocab):
    return torch.tensor([vocab.get(word, vocab["<UNK>"]) for word in text.split()], dtype=torch.long)

# Dataset
class HopeDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = [text_to_tensor(text, vocab) for text in texts]
        self.labels = torch.tensor(labels.values, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_batch(batch):
    texts, labels = zip(*batch)
    return pad_sequence(texts, batch_first=True, padding_value=0), torch.tensor(labels, dtype=torch.long)

# Loaders
train_dataset = HopeDataset(train_df["clean_text"], train_df["label"], vocab)
dev_dataset = HopeDataset(dev_df["clean_text"], dev_df["label"], vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_batch)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False, collate_fn=collate_batch)

# RNN Model
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100, hidden_dim=128, num_classes=5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.rnn = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        _, h = self.rnn(x)
        return self.fc(h.squeeze(0))

# Training and evaluation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RNNClassifier(len(vocab), 100, 128).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

def train(model, loader, optimizer, criterion, epochs=10):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1} - Loss: {total_loss:.4f}")

def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            output = model(x).cpu()
            preds.extend(torch.argmax(output, dim=1).numpy())
            labels.extend(y.numpy())
    print(classification_report(labels, preds, target_names=list(label_map.keys())))

# Run training and evaluation
train(model, train_loader, optimizer, criterion)
evaluate(model, dev_loader)

# Save model, vocab, and label map
torch.save(model.state_dict(), "rnn_multiclass_model.pt")
with open("rnn_vocab.json", "w") as f:
    json.dump(vocab, f)
with open("label_map.json", "w") as f:
    json.dump(label_map, f)
print("✅ Model saved as 'rnn_multiclass_model.pt'")
print("✅ Vocabulary saved as 'rnn_vocab.json'")
print("✅ Label map saved as 'label_map.json'")


Epoch 1 - Loss: 235.8930
Epoch 2 - Loss: 230.7978
Epoch 3 - Loss: 229.1188
Epoch 4 - Loss: 224.9626
Epoch 5 - Loss: 185.5671
Epoch 6 - Loss: 132.7514
Epoch 7 - Loss: 108.7981
Epoch 8 - Loss: 88.5630
Epoch 9 - Loss: 69.4534
Epoch 10 - Loss: 53.0115
                  precision    recall  f1-score   support

Generalized Hope       0.53      0.43      0.47       467
  Realistic Hope       0.39      0.33      0.36       196
Unrealistic Hope       0.35      0.50      0.41       171
        Not Hope       0.72      0.77      0.75       816
         Sarcasm       0.86      0.83      0.84       252

        accuracy                           0.63      1902
       macro avg       0.57      0.57      0.57      1902
    weighted avg       0.63      0.63      0.62      1902

✅ Model saved as 'rnn_multiclass_model.pt'
✅ Vocabulary saved as 'rnn_vocab.json'
✅ Label map saved as 'label_map.json'


In [6]:
# ReBERTa Multi

In [7]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import classification_report
import numpy as np
import json

# Load dataset
train_df = pd.read_csv("en_train.csv")
dev_df = pd.read_csv("en_dev.csv")

# Label encoding
label_map = {
    "Generalized Hope": 0,
    "Realistic Hope": 1,
    "Unrealistic Hope": 2,
    "Not Hope": 3,
    "Sarcasm": 4
}
train_df["label"] = train_df["multiclass"].map(label_map)
dev_df["label"] = dev_df["multiclass"].map(label_map)

# Save label map
with open("roberta_label_map.json", "w") as f:
    json.dump(label_map, f)

# Tokenizer
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Dataset class
class HopeMulticlassDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True, max_length=max_len)
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = HopeMulticlassDataset(train_df["text"], train_df["label"], tokenizer)
dev_dataset = HopeMulticlassDataset(dev_df["text"], dev_df["label"], tokenizer)

# Model
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=5)

# Metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    report = classification_report(labels, preds, target_names=list(label_map.keys()), output_dict=True)
    return {
        "accuracy": report["accuracy"],
        "precision": report["weighted avg"]["precision"],
        "recall": report["weighted avg"]["recall"],
        "f1": report["weighted avg"]["f1-score"],
    }

# Training arguments
training_args = TrainingArguments(
    output_dir="./roberta_multiclass_model",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    logging_dir="./logs_roberta_multiclass"
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Train, evaluate, and save
trainer.train()
trainer.evaluate()

# Save final model and tokenizer
model.save_pretrained("roberta_multiclass_model")
tokenizer.save_pretrained("roberta_multiclass_model")
print("✅ Model + tokenizer saved to 'roberta_multiclass_model/'")


C:\Users\Ryan\anaconda3\envs\PR\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Ryan\anaconda3\envs\PR\lib\site-packages\transformers\training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\Ryan\AppData\Local\Temp\ipykernel_1988\2159454378.py:76: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [8]:
# GPT2 Multi

In [9]:
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import GPT2Tokenizer, GPT2ForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import classification_report
import numpy as np
import json

# Load dataset
train_df = pd.read_csv("en_train.csv")
dev_df = pd.read_csv("en_dev.csv")

# Encode multiclass labels
label_map = {
    "Generalized Hope": 0,
    "Realistic Hope": 1,
    "Unrealistic Hope": 2,
    "Not Hope": 3,
    "Sarcasm": 4
}
train_df["label"] = train_df["multiclass"].map(label_map)
dev_df["label"] = dev_df["multiclass"].map(label_map)

with open("gpt2_label_map.json", "w") as f:
    json.dump(label_map, f)

# Load GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # GPT2 has no official pad token

# Dataset class
class GPT2MulticlassDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(list(texts), padding=True, truncation=True, max_length=max_len)
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = GPT2MulticlassDataset(train_df["text"], train_df["label"], tokenizer)
dev_dataset = GPT2MulticlassDataset(dev_df["text"], dev_df["label"], tokenizer)

# Load GPT-2 model for classification
model = GPT2ForSequenceClassification.from_pretrained("gpt2", num_labels=5)
model.config.pad_token_id = tokenizer.pad_token_id  # fix warning

# Metrics
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    report = classification_report(labels, preds, target_names=list(label_map.keys()), output_dict=True)
    return {
        "accuracy": report["accuracy"],
        "precision": report["weighted avg"]["precision"],
        "recall": report["weighted avg"]["recall"],
        "f1": report["weighted avg"]["f1-score"],
    }

# Training args
training_args = TrainingArguments(
    output_dir="./gpt2_multiclass_model",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    logging_dir="./logs_gpt2_multiclass"
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# Train and evaluate
trainer.train()
trainer.evaluate()

# Save model and tokenizer
model.save_pretrained("gpt2_multiclass_model")
tokenizer.save_pretrained("gpt2_multiclass_model")
print("✅ GPT-2 model and tokenizer saved to 'gpt2_multiclass_model/'")


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\Ryan\anaconda3\envs\PR\lib\site-packages\transformers\training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
C:\Users\Ryan\AppData\Local\Temp\ipykernel_1988\4122797249.py:77: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [11]:
# BiLSTM + Attention Model

In [17]:
# This code sets up BiLSTM+Attention (DeepSeek-style) for multiclass classification and saves model + vocab + labels

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import pandas as pd
import re
import emoji
import json
from collections import Counter
from sklearn.metrics import classification_report

# Load data
train_df = pd.read_csv("en_train.csv")
dev_df = pd.read_csv("en_dev.csv")

# Text preprocessing
def convert_emojis(text):
    return emoji.demojize(text, delimiters=(" ", " "))

def clean_text(text):
    text = re.sub(r"#USER#", "", text)
    text = convert_emojis(text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', 'USER_MENTION', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

train_df["clean_text"] = train_df["text"].apply(clean_text)
dev_df["clean_text"] = dev_df["text"].apply(clean_text)

# Multiclass label mapping
label_map = {
    "Generalized Hope": 0,
    "Realistic Hope": 1,
    "Unrealistic Hope": 2,
    "Not Hope": 3,
    "Sarcasm": 4
}
train_df["label"] = train_df["multiclass"].map(label_map)
dev_df["label"] = dev_df["multiclass"].map(label_map)

# Save label map
with open("deepseek_label_map.json", "w") as f:
    json.dump(label_map, f)

# Vocabulary
word_counts = Counter()
for sentence in train_df["clean_text"]:
    word_counts.update(sentence.split())

vocab = {word: idx + 2 for idx, (word, _) in enumerate(word_counts.most_common())}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

with open("deepseek_vocab.json", "w") as f:
    json.dump(vocab, f)

def text_to_tensor(text, vocab):
    return torch.tensor([vocab.get(word, vocab["<UNK>"]) for word in text.split()], dtype=torch.long)

# Dataset and collator
class HopeMulticlassDataset(Dataset):
    def __init__(self, texts, labels, vocab):
        self.texts = [text_to_tensor(text, vocab) for text in texts]
        self.labels = torch.tensor(labels.values, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_batch(batch):
    texts, labels = zip(*batch)
    padded_texts = pad_sequence(texts, batch_first=True, padding_value=0)
    return padded_texts, torch.tensor(labels, dtype=torch.long)

train_dataset = HopeMulticlassDataset(train_df["clean_text"], train_df["label"], vocab)
dev_dataset = HopeMulticlassDataset(dev_df["clean_text"], dev_df["label"], vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_batch)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False, collate_fn=collate_batch)

# BiLSTM + Attention Model
class BiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim=100, hidden_dim=128, num_classes=5):
        super(BiLSTMAttention, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.attn = nn.Linear(hidden_dim * 2, 1)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        lstm_out, _ = self.lstm(emb)
        attn_weights = torch.softmax(self.attn(lstm_out).squeeze(-1), dim=1)
        context = torch.sum(lstm_out * attn_weights.unsqueeze(-1), dim=1)
        return self.fc(context)

# Training + Evaluation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BiLSTMAttention(len(vocab), 100, 128, num_classes=5).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_model(model, train_loader, criterion, optimizer, num_epochs=15):
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for texts, labels in train_loader:
            texts, labels = texts.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(texts)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

def evaluate_model(model, dev_loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for texts, labels in dev_loader:
            texts = texts.to(device)
            outputs = model(texts).cpu()
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.numpy())
            all_labels.extend(labels.numpy())
    print(classification_report(all_labels, all_preds, target_names=list(label_map.keys())))

# Train and evaluate
train_model(model, train_loader, criterion, optimizer)
evaluate_model(model, dev_loader)

# Save model
torch.save(model.state_dict(), "deepseek_multiclass_model.pt")
"deepseek_multiclass_model.pt, deepseek_vocab.json, and deepseek_label_map.json saved."


Epoch 1, Loss: 220.4224
Epoch 2, Loss: 152.3252
Epoch 3, Loss: 112.0097
Epoch 4, Loss: 86.7090
Epoch 5, Loss: 62.5007
Epoch 6, Loss: 39.7413
Epoch 7, Loss: 22.8604
Epoch 8, Loss: 12.9445
Epoch 9, Loss: 6.6435
Epoch 10, Loss: 2.5758
Epoch 11, Loss: 1.4220
Epoch 12, Loss: 0.9941
Epoch 13, Loss: 0.7307
Epoch 14, Loss: 0.6258
Epoch 15, Loss: 0.4493
                  precision    recall  f1-score   support

Generalized Hope       0.52      0.57      0.54       467
  Realistic Hope       0.46      0.33      0.38       196
Unrealistic Hope       0.34      0.47      0.39       171
        Not Hope       0.75      0.72      0.73       816
         Sarcasm       0.88      0.83      0.85       252

        accuracy                           0.63      1902
       macro avg       0.59      0.58      0.58      1902
    weighted avg       0.65      0.63      0.64      1902



'deepseek_multiclass_model.pt, deepseek_vocab.json, and deepseek_label_map.json saved.'